In [4]:
import json, os

In [5]:
input_folder = "/home/lamdo/IllinoisRetrievalBenchmark/test_run_gitig_10"

step1_files = os.listdir(os.path.join(input_folder, "step1"))

In [6]:
test_data = []
from collections import Counter
counter = Counter()
for file in step1_files:
    step1 = os.path.join(input_folder, "step1", file)
    step2_1 = os.path.join(input_folder, "step2_1", file)
    step2_2 = os.path.join(input_folder, "step2_2", file)
    step3 = os.path.join(input_folder, "step3", file)

    try:

        with open(step1) as f:
            step1_data = json.load(f)

        with open(step2_1) as f:
            step2_1_data = json.load(f)

        with open(step2_2) as f:
            step2_2_data = json.load(f)
        
        with open(step3) as f:
            step3_data = json.load(f)
    except FileNotFoundError: continue

    url_content_mapper = step2_1_data.get("url_content_mapper")
    keypoints_mapper = step2_2_data.get("keypoints_mapper")
    groundedness_check = step3_data.get("groundedness_check")
    if not groundedness_check or not keypoints_mapper: continue

    visited = set()
    for k, score in groundedness_check.items():
        fact_id, url, kp_id = k.split("--__--")
        fact_id = int(fact_id)
        kp_id = int(kp_id)


        # if fact_id not in visited:
        counter[score] += 1
        # visited.add(fact_id)
        if score: continue


        keypoint = keypoints_mapper[str(fact_id)][kp_id]

        url_content = url_content_mapper[url]["url_content"]
        published_date = url_content_mapper[url]["published_date"]
        lang = url_content_mapper[url]["lang"]

        if not lang or lang == "en": continue

        test_data.append(["Fact: "+ keypoint, "Context Published date: " + str(published_date), "Context Language: " + str(lang), "Context: "+ url_content])


In [7]:
counter

Counter({0: 3096, 1: 450})

In [1]:
import gzip, json

In [2]:
with gzip.open("/scratch/lamdo/wiki_dump/enwiki-20250929-cirrussearch-content.json.gz", 'rt', encoding='utf-8') as f:
    test_data = []
    for idx, line in enumerate(f):
        obj = json.loads(line) 
        if isinstance(obj, dict) and obj.get("source_text") is not None:
            test_data.append(obj)

        if len(test_data) == 100: break

In [3]:
from collections import Counter


counter = Counter()
for line in test_data:
    counter.update([line["content_model"]])

counter

Counter({'wikitext': 100})

In [3]:
test_data[80]

{'template': ['Template:Short description',
  'Template:Pagetype',
  'Template:Main other',
  'Template:Short description/lowercasecheck',
  'Template:First word',
  'Template:SDcat',
  'Template:Politics of Angola',
  'Template:Sidebar with collapsible lists',
  'Template:Politics sidebar title',
  'Template:Hlist',
  'Template:Hlist/styles.css',
  'Template:Politics sidebar below',
  'Template:Flatlist',
  'Template:Elect',
  'Template:Hatnote',
  'Template:Interlanguage link',
  'Template:Separated entries',
  'Template:Reflist',
  'Template:Reflist/styles.css',
  'Template:Cite web',
  'Template:Dead link',
  'Template:Fix',
  'Template:Fix/category',
  'Template:Cite news',
  'Template:Africa in topic',
  'Template:Africa topic',
  'Template:Navbox',
  'Template:Longitem',
  'Template:Nbs',
  'Template:Spaces',
  'Template:Angola topics',
  'Template:Country topics',
  'Template:Small',
  'Template:Bulleted list',
  'Template:Nobold',
  'Template:Nobold/styles.css',
  'Template:Ye

In [3]:
from typing import List, Dict, Any
import re

def get_articletopics_with_scores(weighted_tags: List[str]) -> List[Dict[str, Any]]:
    """
    Extracts all 'classification.prediction.articletopic' tags and returns them
    as a list of dictionaries, each containing the topic path and its score as a float.

    Args:
        weighted_tags: A list of weighted tag strings.

    Returns:
        A list of dictionaries with keys 'topic' (str) and 'score' (float).
    """
    topic_list = []
    
    # Regex to capture the topic path and the score for 'articletopic' tags
    # Group 1: The topic path (e.g., History_and_Society.Politics_and_government)
    # Group 2: The score (e.g., 859)
    # Note: We use the raw score (like '859') and convert it to a float score (0.859) later.
    topic_pattern = re.compile(r'classification\.prediction\.articletopic/(.*?)\|(\d+)$')

    for tag in weighted_tags:
        match = topic_pattern.search(tag)
        if match:
            topic_path = match.group(1)
            raw_score = match.group(2)
            
            # Convert the raw score (e.g., '859') into a float score (e.g., 0.859)
            # by dividing by 1000, as is common in these types of classification scores.
            try:
                score = int(raw_score) / 1000.0
            except ValueError:
                # Should not happen with the regex, but a safe check
                continue
            
            # Append the structured dictionary to the list
            topic_list.append({
                'topic': topic_path,
                'score': score
            })

    return topic_list

# --- Example Usage ---

data = [
    'classification.prediction.articletopic/History_and_Society.Politics_and_government|859',
    'classification.prediction.articletopic/Geography.Regions.Africa.Africa*|996',
    'classification.prediction.articletopic/Geography.Regions.Africa.Central_Africa|899',
    'classification.prediction.articlecountry/Angola|1000',
    'recommendation.link/exists',
    'classification.prediction.drafttopic/Geography.Regions.Africa.Central Africa|961',
    'classification.prediction.drafttopic/History and Society.Politics and government|713',
]

result = get_articletopics_with_scores(data)

In [4]:
result

[{'topic': 'History_and_Society.Politics_and_government', 'score': 0.859},
 {'topic': 'Geography.Regions.Africa.Africa*', 'score': 0.996},
 {'topic': 'Geography.Regions.Africa.Central_Africa', 'score': 0.899}]

In [2]:
import json
from collections import Counter

topic_counter = {}
with open("/home/lamdo/IllinoisRetrievalBenchmark/test_run_gitig_10/step5/attributes.jsonl") as f:
    for line in f:
        line = json.loads(line)
        year = line["wiki_create_timestamp"][:4]
        topics = set([item.split(".")[0] for item in line["topics"]])

        for top in topics:
            if top not in topic_counter: topic_counter[top] = Counter()
            topic_counter[top].update([year])

In [3]:
topic_counter

{'Culture': Counter({'2023': 1052, '2025': 10, '2024': 37}),
 'Geography': Counter({'2024': 14, '2025': 4, '2023': 1352}),
 'History_and_Society': Counter({'2025': 2, '2023': 445, '2024': 3}),
 'STEM': Counter({'2023': 345, '2024': 12})}